# Building Agents That Use Code

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

Alfred is planning a party at the Wayne family mansion and needs your help to ensure everything goes smoothly. To assist him, we'll apply what we've learned about how a multi-step `CodeAgent` operates.


> **Colab adaptation:** Original course notebook, adapted to GroqCloud. Run cells in order. In Colab Secrets, enable `GROQ_API_KEY`. HF publishing and Langfuse cloud tracing are opt-in; local export and OpenTelemetry tracing are included by default. All outputs are generated by your own runtime.


## Let's install the dependencies and configure our inference provider

The original course uses the Hugging Face Inference API. Here we use Groq through an OpenAI-compatible endpoint. Run the installation below.

In [ ]:
%pip install -q "smolagents[openai,toolkit]>=1.22,<2" ddgs markdownify requests

Enable the existing `GROQ_API_KEY` in Colab Secrets (key icon). The following cell checks which models your account can access and prepares the model for all examples. No Hugging Face token is required for Groq inference.

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI
from smolagents import OpenAIModel

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
GROQ_BASE = "https://api.groq.com/openai/v1"
client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE)
available = {m.id for m in client.models.list().data}
# Prefer Qwen to avoid the GPT-OSS native tool-call conflict seen with CodeAgent.
preferred_models = ["qwen/qwen3.8-27b", "openai/gpt-oss-120b", "openai/gpt-oss-20b"]
MODEL_ID = next((m for m in preferred_models if m in available), None)
if MODEL_ID is None:
    raise RuntimeError(f"None of the configured chat models is available. Models: {sorted(available)}")
model_kwargs = {"temperature": 0.2, "max_tokens": 4096}
if MODEL_ID.startswith("openai/gpt-oss"):
    model_kwargs["reasoning_effort"] = "low"
model = OpenAIModel(model_id=MODEL_ID, api_base=GROQ_BASE,
                    api_key=GROQ_API_KEY, **model_kwargs)
print("Groq model selected:", MODEL_ID)
# Optional services: enable only when you explicitly want to use them.
PUBLISH_TO_HUB = False
USE_LANGFUSE_CLOUD = False
HF_REPO_ID = "angiedonatto/AlfredAgent"
LANGFUSE_REGION = "EU"

def optional_secret(name):
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None

Groq model selected: qwen/qwen3.8-27b


## Selecting a Playlist for the Party Using `smolagents`

An important part of a successful party is the music. Alfred needs some help selecting the playlist. Luckily, `smolagents` has got us covered! We can build an agent capable of searching the web. To give the agent access to this tool, we include it in the tool list when creating the agent.

The original course uses `InferenceClientModel`. This adaptation uses the Groq-backed `OpenAIModel` configured above, keeping the same `CodeAgent` and web search tool.

Running an agent is quite straightforward:

In [ ]:
from smolagents import CodeAgent, WebSearchTool

agent = CodeAgent(tools=[WebSearchTool(max_results=5, engine="bing")], model=model, max_steps=5, max_print_outputs_length=6000)

agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the best music recommendations for a party at the Wayne's mansion.                                   │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = web_search(query="best music recommendations for a party at Wayne's mansion")                          
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Best Buy | Official Online Store | Shop Now & Save](https://www.bestbuy.com/)
Shop Best Buy for electronics, computers, appliances, cell phones, video games & more new tech. Store pickup & free
2-day shipping on thousands of items.

[Best Buy Store Locator: Store Hours, Directions & Events](https://www.bestbuy.com/site/store-locator)
Use the Best Buy store locator to find stores in your area. Then, visit each Best Buy store's page to see store 
hours, directions, news, events and more.

[BEST Definition & Meaning - Merriam-Webster](https://www.merriam-webster.com/dictionary/best)
In the best of all possible worlds, no one would be without food and clean water. I have a wonderful family and a 
great job, so I feel that I have the best of both worlds.

[Best - definition of best by The Free Dictionary](https://www.thefreedictionary.com/best)
1. In a most excellent way; most creditably or advantageously. 2. To the greatest degree or extent; most: "He was 
certainly the best hated man in the ship" (W. Somerset Maugham).

[BEST | English meaning - Cambridge Dictionary](https://dictionary.cambridge.org/dictionary/english/best)
best adjective, adverb (HIGHEST QUALITY) of the highest quality, to the greatest degree, in the most effective way,
or being the most suitable or pleasing; superlative of good orwell:

Out: None

[Step 1: Duration 1.13 seconds| Input tokens: 2,207 | Output tokens: 76]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (14 outputs left out)

[Step 5: Duration 1.62 seconds| Input tokens: 16,168 | Output tokens: 619]

Reached max steps.

[Step 6: Duration 23.37 seconds| Input tokens: 18,904 | Output tokens: 1,058]

'Based on my searches, I wasn\'t able to find specific curated playlists for a "party at Wayne\'s mansion." The search results consistently returned information about the 2019 TV series *Wayne* rather than a specific party music guide. However, I can provide helpful recommendations based on the most likely interpretations of "Wayne\'s mansion":\n\n## If referring to **Bruce Wayne (Batman)**:\nA mansion party at Wayne Manor would likely feature:\n- **Classical music** – Beethoven, Mozart, and other orchestral pieces for an elegant, old-money atmosphere\n- **Jazz and big band** – Think 1920s–1940s swing for a Gatsby-esque gala\n- **Electronic/ambient** – For a more modern, high-tech billionaire vibe\n- **Film scores** – Hans Zimmer\'s *The Dark Knight* soundtrack, or *Batman* (1989) by Danny Elfman\n\n## If referring to **Wayne Campbell (Wayne\'s World)**:\nThe 1992 film featured a lot of 90s rock and pop. Great party tracks would include:\n- "I\'m a Believer" – The Monkees\n- "The Power

When you run this example, the output will **display a trace of the workflow steps being executed**. It will also print the corresponding Python code with the message:

```python
 ─ Executing parsed code: ────────────────────────────────────────────────────────────────────────────────────────
  results = web_search(query="best music for a Batman party")                                                      
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
```

After a few steps, you'll see the generated playlist that Alfred can use for the party! 🎵

## Using a Custom Tool to Prepare the Menu

Now that we have selected a playlist, we need to organize the menu for the guests. Again, Alfred can take advantage of `smolagents` to do so. Here, we use the `@tool` decorator to define a custom function that acts as a tool. We'll cover tool creation in more detail later, so for now, we can simply run the code.

As you can see in the example below, we will create a tool using `@tool` decorator and include it in the `tools` list.

In [ ]:
from smolagents import CodeAgent, tool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

agent = CodeAgent(tools=[suggest_menu], model=model, max_steps=4)

agent.run("Prepare a formal menu for the party.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Prepare a formal menu for the party.                                                                            │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  menu = suggest_menu(occasion="formal")                                                                           
  print(menu)                                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
3-course dinner with wine and dessert.

Out: None

[Step 1: Duration 18.52 seconds| Input tokens: 2,226 | Output tokens: 61]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("3-course dinner with wine and dessert.")                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 3-course dinner with wine and dessert.

[Step 2: Duration 20.59 seconds| Input tokens: 4,603 | Output tokens: 125]

'3-course dinner with wine and dessert.'

The agent will run for a few steps until finding the answer.

The menu is ready! 🥗

## Using Python Imports Inside the Agent

We have the playlist and menu ready, but we need to check one more crucial detail: preparation time!

Alfred needs to calculate when everything would be ready if he started preparing now, in case they need assistance from other superheroes.

`smolagents` specializes in agents that write and execute Python code snippets, offering sandboxed execution for security. It supports both open-source and proprietary language models, making it adaptable to various development environments.

**Code execution has strict security measures** - imports outside a predefined safe list are blocked by default. However, you can authorize additional imports by passing them as strings in `additional_authorized_imports`.
For more details on secure code execution, see the official [guide](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution).

When creating the agent, we will use `additional_authorized_imports` to allow for importing the `datetime` module.

In [ ]:
from smolagents import CodeAgent
import numpy as np
import time
import datetime

agent = CodeAgent(tools=[], model=model, additional_authorized_imports=['datetime'], max_steps=4)

agent.run(
    """
    Alfred needs to prepare for the party. Here are the tasks:
    1. Prepare the drinks - 30 minutes
    2. Decorate the mansion - 60 minutes
    3. Set up the menu - 45 minutes
    4. Prepare the music and playlist - 45 minutes

    If we start right now, at what time will the party be ready?
    """
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Alfred needs to prepare for the party. Here are the tasks:                                                      │
│     1. Prepare the drinks - 30 minutes                                                                          │
│     2. Decorate the mansion - 60 minutes                                                                        │
│     3. Set up the menu - 45 minutes                                                                             │
│     4. Prepare the music and playlist - 45 minutes                                                              │
│                                                                                                                 │
│     If we start right now, at what time will the party be ready?                                                │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import datetime                                                                                                  
                                                                                                                   
  # Task durations in minutes                                                                                      
  tasks = {                                                                                                        
      "Prepare the drinks": 30,                                                                                    
      "Decorate the mansion": 60,                                                                                  
      "Set up the menu": 45,                                                                                       
      "Prepare the music and playlist": 45                                                                         
  }                                                                                                                
                                                                                                                   
  # Calculate total duration                                                                                       
  total_minutes = sum(tasks.values())                                                                              
  print(f"Total time required: {total_minutes} minutes")                                                           
                                                                                                                   
  # Get current time                                                                                               
  now = datetime.datetime.now()                                                                                    
  print(f"Current time: {now.strftime('%H:%M:%S')}")                                                               
                                                                                                                   
  # Calculate finish time                                                                                          
  finish_time = now + datetime.timedelta(minutes=total_minutes)                                                    
  print(f"Party ready time: {finish_time.strftime('%H:%M:%S')}")                                                   
                                                                                                                   
  final_answer(finish_time.strftime('%H:%M'))                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Total time required: 180 minutes
Current time: 22:27:46
Party ready time: 01:27:46

Final answer: 01:27

[Step 1: Duration 18.92 seconds| Input tokens: 2,220 | Output tokens: 241]

'01:27'

These examples are just the beginning of what you can do with code agents, and we're already starting to see their utility for preparing the party.
You can learn more about how to build code agents in the [smolagents documentation](https://huggingface.co/docs/smolagents).

`smolagents` specializes in agents that write and execute Python code snippets, offering sandboxed execution for security. It supports both open-source and proprietary language models, making it adaptable to various development environments.

## Sharing Our Custom Party Preparator Agent to the Hub

Wouldn't it be **amazing to share our very own Alfred agent with the community**? By doing so, anyone can easily download and use the agent directly from the Hub, bringing the ultimate party planner of Gotham to their fingertips! Let's make it happen! 🎉

The `smolagents` library makes this possible by allowing you to share a complete agent with the community and download others for immediate use. It's as simple as the following:


**Colab adaptation:** The default is a local export (no hosting fees). To publish your own agent, set `PUBLISH_TO_HUB = True` in the setup cell and add a Hugging Face `HF_TOKEN` with write permission in Colab Secrets. Check your account's Spaces hosting terms before opting in. A Space that runs this Groq model will additionally require an `OPENAI_API_KEY` secret containing your Groq key; never put it in the repository.


In [12]:
from smolagents import CodeAgent, WebSearchTool, VisitWebpageTool, FinalAnswerTool, Tool, tool

@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service

class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic gotham'.")


# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(
    tools=[
        WebSearchTool(max_results=5, engine="bing"),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool()
        ],
    model=model,
    max_steps=6,
    max_print_outputs_length=6000,
    verbosity_level=1
)

agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

agent.run("Use suggest_menu, catering_service_tool and superhero_party_theme_generator to plan a formal villain masquerade party. State that the catering service is simulated.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_idea = superhero_party_theme_generator(category="villain masquerade")                                      
  print("Party idea:", party_idea)                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Party idea: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 0.62 seconds| Input tokens: 2,447 | Output tokens: 79]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (17 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""                                                                                                 
  🎭 Formal Villain Masquerade Party Plan 🎭                                                                       
                                                                                                                   
  📋 Party Theme:                                                                                                  
  "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."                    
                                                                                                                   
  🍽️ Menu:                                                                                                          
  Custom menu for the butler.                                                                                      
                                                                                                                   
  🚚 Catering Service:                                                                                             
  Gotham Catering Co.                                                                                              
  (Note: This catering service is simulated.)                                                                      
                                                                                                                   
  Enjoy your formal villain masquerade party!                                                                      
  """)                                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 
🎭 Formal Villain Masquerade Party Plan 🎭

📋 Party Theme:
"Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains."

🍽️ Menu:
Custom menu for the butler.

🚚 Catering Service:
Gotham Catering Co.
(Note: This catering service is simulated.)

Enjoy your formal villain masquerade party!

[Step 2: Duration 24.94 seconds| Input tokens: 5,374 | Output tokens: 431]

'\n🎭 Formal Villain Masquerade Party Plan 🎭\n\n📋 Party Theme:\n"Gotham Rogues\' Ball: A mysterious masquerade where guests dress as classic Batman villains."\n\n🍽️ Menu:\nCustom menu for the butler.\n\n🚚 Catering Service:\nGotham Catering Co.\n(Note: This catering service is simulated.)\n\nEnjoy your formal villain masquerade party!\n'

In [21]:
from pathlib import Path
import shutil

# Dynamically inject the source code of the custom tool class if it cannot be found in IPython history
try:
    from smolagents import Tool
    for tool_name, tool_instance in list(agent.tools.items()):
        if isinstance(tool_instance, Tool) and tool_instance.__class__.__name__ == "SuperheroPartyThemeTool":
            tool_instance.__class__.__source__ = """class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = \"\"\"This tool suggests creative superhero-themed party ideas based on a category. It returns a unique party theme idea.\"\"\"

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets."
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic gotham'.")"""
except Exception as e:
    print("Warning during metadata injection:", e)

EXPORT_DIR = Path("AlfredAgent_export")
agent.save(EXPORT_DIR)
# Verify that the serialized export contains no Groq credential before sharing.
for path in EXPORT_DIR.rglob("*"):
    if path.is_file():
        if GROQ_API_KEY.encode() in path.read_bytes():
            raise RuntimeError(f"Secret found in export: {path.name}. Do not share.")
archive = shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=EXPORT_DIR)
print("Local export:", archive)

published = False
hf_token = optional_secret("HF_TOKEN") if PUBLISH_TO_HUB else None
if PUBLISH_TO_HUB and hf_token:
    try:
        agent.push_to_hub(HF_REPO_ID, token=hf_token, private=True)
        published = True
        print("Published to:", f"https://huggingface.co/spaces/{HF_REPO_ID}")
    except Exception as exc:
        print("Hub publication failed:", type(exc).__name__)
        print("The local export is still available. Check permissions and Spaces plan.")
elif PUBLISH_TO_HUB:
    print("HF_TOKEN missing; publication not performed.")
else:
    print("Hub publication not requested; local export only.")

This agent has step_callbacks: they will be ignored by this method. 1

Local export: /content/AlfredAgent_export.zip
Hub publication not requested; local export only.


To download the agent again, use `CodeAgent.from_hub()` on your own published Space. If publication is not enabled, the same serialized agent is loaded from the local export using `CodeAgent.from_folder()`. Both operations reuse Groq credentials only in memory. Never execute arbitrary remote tools without inspecting their code.

In [22]:
from unittest.mock import patch
from smolagents import CodeAgent

# OpenAIModel deserialization looks for OPENAI_API_KEY; scoped to this load only.
with patch.dict(os.environ, {"OPENAI_API_KEY": GROQ_API_KEY}):
    if published:
        alfred_agent = CodeAgent.from_hub(
            HF_REPO_ID, token=hf_token, trust_remote_code=True, model=model
        )
    else:
        alfred_agent = CodeAgent.from_folder(EXPORT_DIR, model=model)
print("Agent loaded from", "your Hub repository" if published else "local export")

Agent loaded from local export


In [23]:
alfred_agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_theme = superhero_party_theme_generator(category="villain masquerade")                                     
  print("Party theme idea:", party_theme)                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Party theme idea: Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 0.45 seconds| Input tokens: 2,447 | Output tokens: 78]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (18 outputs left out)

[Step 6: Duration 8.90 seconds| Input tokens: 17,729 | Output tokens: 714]

Reached max steps.

[Step 7: Duration 13.68 seconds| Input tokens: 19,090 | Output tokens: 1,780]

'Since the web search tool is not returning results, I\'ll craft a curated playlist recommendation based on my knowledge of music that fits a "Gotham Rogues\' Ball: Villain Masquerade" theme at Wayne Manor.\n\n---\n\n## 🎭 **Gotham Rogues\' Ball – Villain Masquerade Playlist**\n\n### **Act I: The Grand Entrance (Dramatic & Ominous)**\n1. **"Main Title" – *The Dark Knight* (Hans Zimmer & James Newton Howard)** – Sets the brooding, epic tone.\n2. **"Villain" – *The Dark Knight Rises* (Hans Zimmer)** – Bane\'s theme; perfect for dramatic entrances.\n3. **"Joker" – *The Dark Knight* (Hans Zimmer)** – Chaotic, unsettling, iconic.\n4. **"Prelude in E Minor, Op. 23" – Chopin** – Classic masquerade elegance with a dark edge.\n5. **"Ride of the Valkyries" – Wagner** – Theatrical and powerful for a villain\'s grand reveal.\n\n### **Act II: The Masquerade Dance (Elegant & Sinister)**\n6. **"The Entertainer" – Scott Joplin** – A twist on classic masquerade waltz energy.\n7. **"Gymnopédie No. 1" – E

What's also exciting is that shared agents are directly available as Hugging Face Spaces, allowing you to interact with them in real-time. You can explore other agents [here](https://huggingface.co/spaces/davidberenstein1957/smolagents-and-tools).

For example, the _AlfredAgent_ is available [here](https://huggingface.co/spaces/sergiopaniego/AlfredAgent).

### Inspecting Our Party Preparator Agent with OpenTelemetry and Langfuse 📡

Full trace can be found [here](https://cloud.langfuse.com/project/cm7bq0abj025rad078ak3luwi/traces/995fc019255528e4f48cf6770b0ce27b?timestamp=2025-02-19T10%3A28%3A36.929Z).

As Alfred fine-tunes the Party Preparator Agent, he's growing weary of debugging its runs. Agents, by nature, are unpredictable and difficult to inspect. But since he aims to build the ultimate Party Preparator Agent and deploy it in production, he needs robust traceability for future monitoring and analysis.  

Once again, `smolagents` comes to the rescue! It embraces the [OpenTelemetry](https://opentelemetry.io/) standard for instrumenting agent runs, allowing seamless inspection and logging. With the help of [Langfuse](https://langfuse.com/) and the `SmolagentsInstrumentor`, Alfred can easily track and analyze his agent’s behavior.  

Setting it up is straightforward!  

First, we need to install the necessary dependencies:  

In [24]:
%pip install -q "smolagents[telemetry]>=1.22,<2" opentelemetry-sdk opentelemetry-exporter-otlp-proto-http openinference-instrumentation-smolagents

Next, Alfred has already created an account on Langfuse and has his API keys ready. If you haven’t done so yet, you can sign up for Langfuse Cloud [here](https://cloud.langfuse.com/) or explore [alternatives](https://huggingface.co/docs/smolagents/tutorials/inspect_runs).  

Once you have your API keys, they need to be properly configured as follows:

**Colab adaptation:** Local OpenTelemetry tracing is the default and requires no additional account. To send traces to Langfuse, set `USE_LANGFUSE_CLOUD = True` in the setup cell and enable `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` in Colab Secrets. Select the region of your project. Cloud tracing transmits the agent's prompts, actions and results to Langfuse.


In [25]:
import base64
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memory_exporter = InMemorySpanExporter()
langfuse_enabled = False
if USE_LANGFUSE_CLOUD:
    public_key = optional_secret("LANGFUSE_PUBLIC_KEY")
    secret_key = optional_secret("LANGFUSE_SECRET_KEY")
    if public_key and secret_key:
        if LANGFUSE_REGION not in ("EU", "US"):
            raise ValueError("LANGFUSE_REGION must be EU or US")
        langfuse_auth = base64.b64encode(f"{public_key}:{secret_key}".encode()).decode()
        langfuse_host = "https://us.cloud.langfuse.com" if LANGFUSE_REGION == "US" else "https://cloud.langfuse.com"
        langfuse_enabled = True
    else:
        print("Langfuse keys missing. Local tracing only.")
print("Trace destination:", "local + Langfuse" if langfuse_enabled else "local memory")

Trace destination: local memory


Finally, Alfred is ready to initialize the `SmolagentsInstrumentor` and start tracking his agent's performance.  

In [26]:
from opentelemetry.sdk.trace import TracerProvider
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
if langfuse_enabled:
    cloud_exporter = OTLPSpanExporter(
        endpoint=f"{langfuse_host}/api/public/otel/v1/traces",
        headers={"Authorization": f"Basic {langfuse_auth}"}, timeout=15,
    )
    trace_provider.add_span_processor(SimpleSpanProcessor(cloud_exporter))

instrumentor = SmolagentsInstrumentor()
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()
instrumentor.instrument(tracer_provider=trace_provider)
print("OpenTelemetry instrumentation enabled.")

OpenTelemetry instrumentation enabled.


Alfred is now connected 🔌! The runs from `smolagents` are being logged in Langfuse, giving him full visibility into the agent's behavior. With this setup, he's ready to revisit previous runs and refine his Party Preparator Agent even further.  

In this adaptation, all runs are traced locally first; Langfuse receives traces only when you explicitly enable cloud export. A configured exporter alone does not establish successful delivery.


In [27]:
memory_exporter.clear()
alfred_agent.run("Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")
trace_provider.force_flush()
spans = memory_exporter.get_finished_spans()
print(f"Finished OpenTelemetry spans: {len(spans)}")
for span in spans:
    duration_ms = (span.end_time - span.start_time) / 1_000_000
    print(f"{span.name}: {duration_ms:.1f} ms | {span.status.status_code.name}")
if langfuse_enabled:
    print("Check your own Langfuse dashboard to confirm trace delivery:", langfuse_host)
else:
    print("Local tracing only; no trace was sent to Langfuse.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Give me best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme        │
│                                                                                                                 │
╰─ OpenAIModel - qwen/qwen3.8-27b ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_theme = superhero_party_theme_generator(category="villain masquerade")                                     
  print(party_theme)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.

Out: None

[Step 1: Duration 150.98 seconds| Input tokens: 2,447 | Output tokens: 73]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (18 outputs left out)

[Step 6: Duration 3.03 seconds| Input tokens: 17,779 | Output tokens: 643]

Reached max steps.

[Step 7: Duration 1.57 seconds| Input tokens: 19,108 | Output tokens: 1,366]

Finished OpenTelemetry spans: 20
OpenAIModel.generate: 150967.0 ms | OK
SuperheroPartyThemeTool: 0.2 ms | OK
Step 1: 150980.4 ms | OK
OpenAIModel.generate: 516.7 ms | OK
WebSearchTool: 134.6 ms | ERROR
Step 2: 669.0 ms | ERROR
OpenAIModel.generate: 595.4 ms | OK
WebSearchTool: 118.5 ms | ERROR
Step 3: 730.7 ms | ERROR
OpenAIModel.generate: 494.0 ms | OK
WebSearchTool: 114.9 ms | ERROR
Step 4: 626.3 ms | ERROR
OpenAIModel.generate: 479.9 ms | OK
WebSearchTool: 113.2 ms | ERROR
Step 5: 610.7 ms | ERROR
OpenAIModel.generate: 2884.9 ms | OK
WebSearchTool: 126.4 ms | ERROR
Step 6: 3028.4 ms | ERROR
OpenAIModel.generate: 1562.1 ms | OK
CodeAgent.run: 158258.0 ms | OK
Local tracing only; no trace was sent to Langfuse.


In [28]:
from ddgs import DDGS

results = DDGS(timeout=15).text(
    "Batman villain music playlist",
    max_results=5,
    backend="auto"
)

for result in results:
    print(result["title"])
    print(result["href"])
    print()

Batman villains playlist - YouTubeBatman Villains Theme Songs (In my opinion) - YouTubeBatman Villain Theme Songs - YouTube MusicBatman Villains Playlist Vol. 2 - Arkham Asylum - YouTube MusicBatman sound edits - playlist by 30h6kbhcdxmg9ybpf87pc9g7c ...Stream BATMAN | Listen to villain arc playlist online for ...
https://www.youtube.com/watch?v=b26mljgLQhw

Batman Villains Theme Songs (In my opinion) - YouTube
https://www.youtube.com/playlist?list=PLF5H5JJXtS5vaaCic2Gf7nP0HjDNZpO-4

Batman Villain Theme Songs - YouTube Music
https://music.youtube.com/playlist?list=PLd7dMC1Iji2415JC2JQmIKkEAGlTr1TXb

Batman Villains Playlist Vol. 2 - Arkham Asylum - YouTube Music
https://music.youtube.com/watch?v=9gMp-ODT-CY

Batman sound edits - playlist by 30h6kbhcdxmg9ybpf87pc9g7c ...
https://open.spotify.com/playlist/2UkurFjcli5rGBHN2viSTQ



**Troubleshooting:** The original web search tool failed to retrieve results in Google Colab. As an alternative, DDGS with automatic backend selection was tested successfully, retrieving real music recommendations from YouTube and Spotify. The alternative search was validated independently but was not yet integrated into the agent.

Alfred can now access this logs [here](https://cloud.langfuse.com/project/cm7bq0abj025rad078ak3luwi/traces/995fc019255528e4f48cf6770b0ce27b?timestamp=2025-02-19T10%3A28%3A36.929Z) to review and analyze them.  

Meanwhile, the [suggested playlist](https://open.spotify.com/playlist/0gZMMHjuxMrrybQ7wTMTpw) sets the perfect vibe for the party preparations. Cool, right? 🎶


**Before submitting:** Save this notebook with your own execution outputs. The local export is not evidence of a published Space, and local tracing is not evidence of Langfuse cloud delivery. Record which optional steps you actually completed.
